<a href="https://colab.research.google.com/github/MuhammadOkasha004/flyrank-ml-internship-work/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadOkasha004/flyrank-ml-internship-work/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

**Load dataset with all columns**

In [1]:


import duckdb
from huggingface_hub import get_token

# 1. Setup Auth Token & Connection
token = get_token()
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{token}');")

rel = "hf://datasets/FlyRank/internship-warehouse"

# 2. Query to Fetch EXACTLY 1 Row
query_single_row = f"""
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
LIMIT 1;
"""

# 3. Execute Query
df_single = con.sql(query_single_row).df()

# 4. Print All 31 Column Names strictly as a Clean List
print("=" * 65)
print("ALL 31 COLUMNS IN RAW DATASET:")
print("=" * 65)

for idx, col_name in enumerate(df_single.columns, 1):
    print(f"{idx:02d}. {col_name}")

print("\n" + "=" * 65)
print("SINGLE ROW DATA SAMPLE:")
print("=" * 65)
display(df_single)




ALL 31 COLUMNS IN RAW DATASET:
01. report_date
02. client_hash_id
03. content_hash_id
04. client_has_gsc
05. client_has_ga4
06. gsc_data_available
07. ga4_data_available
08. gsc_impressions
09. gsc_clicks
10. gsc_sum_position
11. gsc_avg_position
12. ga4_pageviews
13. ga4_sessions
14. ga4_users
15. ga4_engaged_sessions
16. ga4_total_engagement_sec
17. sessions_organic
18. sessions_direct
19. sessions_referral
20. sessions_social
21. sessions_paid
22. sessions_ai
23. ai_chatgpt
24. ai_perplexity
25. ai_gemini
26. ai_copilot
27. ai_claude
28. ai_meta
29. ai_other
30. scroll_events
31. month

SINGLE ROW DATA SAMPLE:


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


**Load Dataset of month1,2,3**

In [2]:
import duckdb
from huggingface_hub import get_token

# 1. Setup Auth Token & Connection
token = get_token()
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{token}');")

# Fast Threading Settings
con.execute("SET preserve_insertion_order = false;")
con.execute("SET threads = 4;")

rel = "hf://datasets/FlyRank/internship-warehouse"

# 2. Query: Group BY content_hash_id, client_hash_id, month
page_month_query = f"""
SELECT
    content_hash_id,
    client_hash_id,
    month,
    SUM(gsc_clicks) as gsc_clicks,
    SUM(gsc_impressions) as gsc_impressions,
    ROUND(AVG(gsc_avg_position), 2) as gsc_avg_position,
    SUM(ga4_total_engagement_sec) as ga4_total_engagement_sec,
    SUM(sessions_organic) as sessions_organic,
    SUM(sessions_ai) as sessions_ai

FROM (
    -- Scanning ONLY strictly selected 9 columns for Months 1, 2, and 3 (Jan, Feb, Mar 2026)
    SELECT
        content_hash_id,
        client_hash_id,
        month,
        gsc_clicks,
        gsc_impressions,
        gsc_avg_position,
        ga4_total_engagement_sec,
        sessions_organic,
        sessions_ai
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-0*/*.parquet')
    WHERE gsc_data_available = TRUE
      AND month IN ('2026-01', '2026-02', '2026-03')  -- Strictly Months 1, 2, and 3
)
GROUP BY content_hash_id, client_hash_id, month
ORDER BY content_hash_id, month;
"""

# 3. Execute and Load DataFrame
print("Executing clean Page-Month grain query...")
df_page_month = con.sql(page_month_query).df()

# 4. Output Shape & Verification
print("=" * 65)
print("PAGE-MONTH GRAIN DATASET LOADED SUCCESSFULLY!")
print("=" * 65)
print("Dataset Shape (Rows, Columns):", df_page_month.shape)
print("Exact Columns Count:", df_page_month.shape[1])

print("\nFirst 6 Rows Preview (Notice Page A with Month 1, Month 2, Month 3):")
display(df_page_month.head(6))

Executing clean Page-Month grain query...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

PAGE-MONTH GRAIN DATASET LOADED SUCCESSFULLY!
Dataset Shape (Rows, Columns): (451841, 9)
Exact Columns Count: 9

First 6 Rows Preview (Notice Page A with Month 1, Month 2, Month 3):


,content_hash_id,client_hash_id,month,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_total_engagement_sec,sessions_organic,sessions_ai
0,content_000005d4ced12088,client_9958f0a7ae1df715,2026-01,0.0,15.0,74.23,0.0,0.0,0.0
1,content_000005d4ced12088,client_9958f0a7ae1df715,2026-02,0.0,24.0,87.72,0.0,0.0,0.0
2,content_000005d4ced12088,client_9958f0a7ae1df715,2026-03,0.0,86.0,72.85,0.0,0.0,0.0
3,content_00007bd2985b77c3,client_73cda7b4e4f265ea,2026-01,0.0,10.0,6.47,NaN,NaN,NaN
4,content_00007bd2985b77c3,client_73cda7b4e4f265ea,2026-02,0.0,16.0,3.19,NaN,NaN,NaN
5,content_00007bd2985b77c3,client_73cda7b4e4f265ea,2026-03,0.0,47.0,5.27,0.0,0.0,0.0


In [3]:
display(df_page_month.tail())

,content_hash_id,client_hash_id,month,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_total_engagement_sec,sessions_organic,sessions_ai
451836,content_ffffc58385523096,client_e547b89c05043229,2026-02,17.0,3319.0,5.35,319.0,9.0,0.0
451837,content_ffffc58385523096,client_e547b89c05043229,2026-03,37.0,2482.0,4.07,212.0,33.0,2.0
451838,content_fffff09da8a25da6,client_73cda7b4e4f265ea,2026-01,1.0,4569.0,1.11,NaN,NaN,NaN
451839,content_fffff09da8a25da6,client_73cda7b4e4f265ea,2026-02,2.0,2813.0,0.35,NaN,NaN,NaN
451840,content_fffff09da8a25da6,client_73cda7b4e4f265ea,2026-03,0.0,563.0,1.41,0.0,0.0,0.0


**Handle Missing Value**

In [4]:
# 1. Fill volume metrics with 0
zero_fill_cols = [
    'gsc_clicks',
    'gsc_impressions',
    'ga4_total_engagement_sec',
    'sessions_organic',
    'sessions_ai'
]
df_page_month[zero_fill_cols] = df_page_month[zero_fill_cols].fillna(0)

# 2. Fill position/ranking metric with 100.0 (unranked)
df_page_month['gsc_avg_position'] = df_page_month['gsc_avg_position'].fillna(100.0)

# Preview cleaned dataframe
display(df_page_month.head())

,content_hash_id,client_hash_id,month,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_total_engagement_sec,sessions_organic,sessions_ai
0,content_000005d4ced12088,client_9958f0a7ae1df715,2026-01,0.0,15.0,74.23,0.0,0.0,0.0
1,content_000005d4ced12088,client_9958f0a7ae1df715,2026-02,0.0,24.0,87.72,0.0,0.0,0.0
2,content_000005d4ced12088,client_9958f0a7ae1df715,2026-03,0.0,86.0,72.85,0.0,0.0,0.0
3,content_00007bd2985b77c3,client_73cda7b4e4f265ea,2026-01,0.0,10.0,6.47,0.0,0.0,0.0
4,content_00007bd2985b77c3,client_73cda7b4e4f265ea,2026-02,0.0,16.0,3.19,0.0,0.0,0.0


**Filter Inactive Rows**

In [5]:
# Traffic metrics list
traffic_cols = ['gsc_clicks', 'gsc_impressions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_ai']

# Inactive (dead) rows filter out karna
df_page_month = df_page_month[df_page_month[traffic_cols].sum(axis=1) > 0]

# Shape verify karna
display(df_page_month.shape)
display(df_page_month.head())

(451841, 9)

,content_hash_id,client_hash_id,month,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_total_engagement_sec,sessions_organic,sessions_ai
0,content_000005d4ced12088,client_9958f0a7ae1df715,2026-01,0.0,15.0,74.23,0.0,0.0,0.0
1,content_000005d4ced12088,client_9958f0a7ae1df715,2026-02,0.0,24.0,87.72,0.0,0.0,0.0
2,content_000005d4ced12088,client_9958f0a7ae1df715,2026-03,0.0,86.0,72.85,0.0,0.0,0.0
3,content_00007bd2985b77c3,client_73cda7b4e4f265ea,2026-01,0.0,10.0,6.47,0.0,0.0,0.0
4,content_00007bd2985b77c3,client_73cda7b4e4f265ea,2026-02,0.0,16.0,3.19,0.0,0.0,0.0


**Get Engineered Features**

In [6]:
import pandas as pd
import numpy as np

# 1. Dataset ko Content ID aur Month ke hisab se sort karna
df_page_month = df_page_month.sort_values(by=['content_hash_id', 'month']).reset_index(drop=True)

# 2. Direct Monthly Ratios (Single Column Names)
df_page_month['ctr'] = df_page_month['gsc_clicks'] / (df_page_month['gsc_impressions'] + 1)

df_page_month['sec_per_click'] = df_page_month['ga4_total_engagement_sec'] / (df_page_month['gsc_clicks'] + 1)

total_sessions = df_page_month['sessions_organic'] + df_page_month['sessions_ai']
df_page_month['ai_share'] = df_page_month['sessions_ai'] / (total_sessions + 1)

# 3. Lag / Shift Calculations Per Page (Velocity & Position Drift)
df_page_month['click_vel'] = df_page_month.groupby('content_hash_id')['gsc_clicks'].diff().fillna(0)

df_page_month['imp_vel'] = df_page_month.groupby('content_hash_id')['gsc_impressions'].diff().fillna(0)

# Position Drift (Pos_M2 - Pos_M1)
df_page_month['pos_drift'] = df_page_month.groupby('content_hash_id')['gsc_avg_position'].diff().fillna(0)

# 4. Result Preview
display(df_page_month.shape)
display(df_page_month.tail())
df_page_month.columns.to_list()

(451841, 15)

,content_hash_id,client_hash_id,month,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_total_engagement_sec,sessions_organic,sessions_ai,ctr,sec_per_click,ai_share,click_vel,imp_vel,pos_drift
451836,content_ffffc58385523096,client_e547b89c05043229,2026-02,17.0,3319.0,5.35,319.0,9.0,0.0,0.005120,17.722222,0.000000,-16.0,-3963.0,1.17
451837,content_ffffc58385523096,client_e547b89c05043229,2026-03,37.0,2482.0,4.07,212.0,33.0,2.0,0.014901,5.578947,0.055556,20.0,-837.0,-1.28
451838,content_fffff09da8a25da6,client_73cda7b4e4f265ea,2026-01,1.0,4569.0,1.11,0.0,0.0,0.0,0.000219,0.000000,0.000000,0.0,0.0,0.00
451839,content_fffff09da8a25da6,client_73cda7b4e4f265ea,2026-02,2.0,2813.0,0.35,0.0,0.0,0.0,0.000711,0.000000,0.000000,1.0,-1756.0,-0.76
451840,content_fffff09da8a25da6,client_73cda7b4e4f265ea,2026-03,0.0,563.0,1.41,0.0,0.0,0.0,0.000000,0.000000,0.000000,-2.0,-2250.0,1.06


['content_hash_id',
 'client_hash_id',
 'month',
 'gsc_clicks',
 'gsc_impressions',
 'gsc_avg_position',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_ai',
 'ctr',
 'sec_per_click',
 'ai_share',
 'click_vel',
 'imp_vel',
 'pos_drift']

**Categorical Handling**

In [7]:
# Direct replacement mapping
month_map = {'2026-01': 1, '2026-02': 2, '2026-03': 3}

# Direct value replace
df_page_month['month'] = df_page_month['month'].replace(month_map)

# Verify results
display(df_page_month['month'].value_counts())
display(df_page_month.head())

/tmp/ipykernel_2609/2534648110.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_page_month['month'] = df_page_month['month'].replace(month_map)


,count
month,
3,176738
2,153559
1,121544


,content_hash_id,client_hash_id,month,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_total_engagement_sec,sessions_organic,sessions_ai,ctr,sec_per_click,ai_share,click_vel,imp_vel,pos_drift
0,content_000005d4ced12088,client_9958f0a7ae1df715,1,0.0,15.0,74.23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00
1,content_000005d4ced12088,client_9958f0a7ae1df715,2,0.0,24.0,87.72,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.0,13.49
2,content_000005d4ced12088,client_9958f0a7ae1df715,3,0.0,86.0,72.85,0.0,0.0,0.0,0.0,0.0,0.0,0.0,62.0,-14.87
3,content_00007bd2985b77c3,client_73cda7b4e4f265ea,1,0.0,10.0,6.47,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00
4,content_00007bd2985b77c3,client_73cda7b4e4f265ea,2,0.0,16.0,3.19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,-3.28


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### 📊 Feature Notes & Data Leakage Matrix (13 Predictive Features)

| Feature Name | Feature Type | Meaning / Description | Missing Value Handling | Available BEFORE Prediction? |
| :--- | :--- | :--- | :--- | :--- |
| **`month`** | Encoded Time | Numerical time index (`1`, `2`, `3`). | Default value `1` / Non-null | **YES** (Current time step is known) |
| **`gsc_clicks`** | Base Metric | Google Search Clicks (Current Month). | `fillna(0)` | **YES** (Known at month-end observation) |
| **`gsc_impressions`** | Base Metric | Total Search Impressions. | `fillna(0)` | **YES** (Known at month-end observation) |
| **`gsc_avg_position`** | Base Metric | Average Search Rank. | `fillna(100.0)` | **YES** (Known at month-end observation) |
| **`ga4_total_engagement_sec`** | Base Metric | Total User Engagement (Seconds). | `fillna(0)` | **YES** (Known at month-end observation) |
| **`sessions_organic`** | Base Metric | Organic Search Sessions. | `fillna(0)` | **YES** (Known at month-end observation) |
| **`sessions_ai`** | Base Metric | AI Referral / Search Sessions. | `fillna(0)` | **YES** (Known at month-end observation) |
| **`ctr`** | Derived Feature | `clicks / (impressions + 1)` | Handled via formula (`+ 1`) | **YES** (Derived from historical base metrics) |
| **`sec_per_click`** | Derived Feature | `engagement_sec / (clicks + 1)` | Handled via formula (`+ 1`) | **YES** (Derived from historical base metrics) |
| **`ai_share`** | Derived Feature | `sessions_ai / (total_sessions + 1)` | Handled via formula (`+ 1`) | **YES** (Derived from historical base metrics) |
| **`click_vel`** | Derived Feature | Monthly Click Difference ($M_t - M_{t-1}$). | `fillna(0)` (Month 1 base) | **YES** (Requires previous month historical data) |
| **`imp_vel`** | Derived Feature | Impression Difference ($M_t - M_{t-1}$). | `fillna(0)` (Month 1 base) | **YES** (Requires previous month historical data) |
| **`pos_drift`** | Derived Feature | Rank Movement ($M_t - M_{t-1}$). | `fillna(0)` (Month 1 base) | **YES** (Requires previous month historical data) |

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.